In [24]:
''' U-Net without cropping. Each image is resized to 512x512.

The test, validation and training sets are stored separately according to the split decided by the paper.
Will use pillow instead of openCV
'''
import os, json, random
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CROP, BATCH, EPOCHS, LR = 256, 8, 100, 1e-2
SAMPLES_PER_IMAGE = 8           # random crops drawn per image per epoch
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)








In [25]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset

SIZE = (256, 256)  # (width, height)

dirs = {
        "train": (r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\images\train",
                  r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\masks\0\train"),


        "val":   (r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\images\val",
                  r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\masks\0\val"),


        "test":  (r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\images\test",
                  r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi\masks\0\test"),
    }

class SegDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=SIZE):
        self.size = size
        images = {p.stem: p for p in Path(image_dir).iterdir() if p.is_file()}
        masks = {p.stem.removesuffix('_mask'): p for p in Path(mask_dir).iterdir() if p.is_file()}

        # pair files by name without extension, so img.png <-> img.tif still match
        self.names = sorted(images.keys() & masks.keys())
        assert self.names, "No matching image/mask names found"
        if len(self.names) != len(images) or len(self.names) != len(masks):
            print(f"Warning: {len(images)} images, {len(masks)} masks, "
                  f"{len(self.names)} matched")
        self.images = [images[n] for n in self.names]
        self.masks = [masks[n] for n in self.names]

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("L")  # grayscale SEM
        mask = Image.open(self.masks[idx])

        # If your mask must first be combined with the image or converted
        # (e.g. a matrix you multiply/threshold), do it here, before resizing.

        # bilinear for the image, nearest for the mask so labels aren't blended
        image = image.resize(self.size, Image.NEAREST)  # or Image.BILINEAR
        mask = mask.resize(self.size, Image.NEAREST)

        image = torch.from_numpy(np.array(image, dtype=np.float32) / 255.0).unsqueeze(0)  # (1,H,W)
        mask = torch.from_numpy(np.array(mask)>127).long() 
        
                                         # (H,W)
        # If masks are stored as 0/255, use: mask = (mask > 127).long()
        return image, mask


def make_loaders(dirs, batch_size=8, num_workers=0, limits = None):
    """dirs: {"train": (image_dir, mask_dir), "val": (...), "test": (...)}"""
    limits = limits or {}
    loaders = {}
    for split, (image_dir, mask_dir) in dirs.items():
        ds = SegDataset(image_dir, mask_dir)
        if split in limits:
            ds = Subset(ds, range(min(limits[split], len(ds))))
        loaders[split] = DataLoader(
            ds,
            batch_size,
            shuffle=(split == "train"),  # only shuffle training data
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders["train"], loaders["val"], loaders["test"]


if __name__ == "__main__":  # required on Windows when num_workers > 0

    train, val, test = make_loaders(dirs, batch_size=BATCH, num_workers=0, limits={"train": 100, "val": 20, "test": 20})


    for name, loader in [("train", train), ("val", val), ("test", test)]:
        print(name, len(loader.dataset), "samples")
    images, masks = next(iter(train))
    print(images.shape, images.dtype)   # torch.Size([8, 1, 512, 512]) torch.float32
    print(masks.shape, masks.dtype, masks.unique())


train 100 samples
val 20 samples
test 20 samples
torch.Size([8, 1, 256, 256]) torch.float32
torch.Size([8, 256, 256]) torch.int64 tensor([0, 1])


In [26]:
# --------------------------------------------------------------------- model
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]   # 32, 64, 128, 256, 512
        self.enc1 = DoubleConv(in_ch, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.bottleneck = DoubleConv(c[3], c[4])
        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(c[4], c[3], 2, stride=2)
        self.dec4 = DoubleConv(c[3] * 2, c[3])
        self.up3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
        self.dec3 = DoubleConv(c[2] * 2, c[2])
        self.up2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
        self.dec2 = DoubleConv(c[1] * 2, c[1])
        self.up1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
        self.dec1 = DoubleConv(c[0] * 2, c[0])
        self.head = nn.Conv2d(c[0], out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)          # raw logits; sigmoid is applied in loss / eval


# ---------------------------------------------------------------------- loss
bce = torch.nn.BCEWithLogitsLoss()

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)                          # (B,1,H,W)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()

def loss_fn(logits, y):
    target = y.unsqueeze(1).float()                    # (B,H,W) long -> (B,1,H,W) float
    return bce(logits, target) + dice_loss(logits, target)

In [27]:


# ------------------------------------------------------------------- metrics
@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    model.eval()
    tp = fp = fn = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        assert y.max() <= 1 and y.min() >= 0, f"bad mask values: {y.unique()}"
        pred = (torch.sigmoid(model(x)) > 0.5).squeeze(1)    # (B,H,W) bool
        true = y.bool()                                       # (B,H,W) bool
        assert pred.shape == true.shape, (pred.shape, true.shape)
        tp += (pred & true).sum().item()
        fp += (pred & ~true).sum().item()
        fn += (~pred & true).sum().item()
    iou = tp / (tp + fp + fn + 1e-8)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return iou, dice




In [28]:

# ------------------------------------------------------------------ training
def main():
    train_loader, val_loader, test_loader = make_loaders(
    dirs, batch_size=BATCH, num_workers=0,
    limits={"train": 16, "val": 8, "test": 8},   # remove limits for the real run
    )
    model = UNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5)

    best = 0.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
        val_iou, val_dice = evaluate(model, val_loader)
        sched.step(val_iou)
        print(f"epoch {epoch:3d} | train loss {total / len(train_loader.dataset):.4f} "
              f"| val IoU {val_iou:.4f} | val Dice {val_dice:.4f}")
        if val_iou > best:
            best = val_iou
            torch.save(model.state_dict(), "unet_best.pt")

    # final report: use the checkpoint chosen on validation, touch test once
    model.load_state_dict(torch.load("unet_best.pt", map_location=DEVICE))
    test_iou, test_dice = evaluate(model, test_loader)
    print(f"TEST | IoU {test_iou:.4f} | Dice {test_dice:.4f}")


if __name__ == "__main__":
    main()

epoch   1 | train loss 1.5713 | val IoU 0.0690 | val Dice 0.1290
epoch   2 | train loss 1.3895 | val IoU 0.0690 | val Dice 0.1290
epoch   3 | train loss 1.2737 | val IoU 0.0690 | val Dice 0.1290
epoch   4 | train loss 1.1496 | val IoU 0.0690 | val Dice 0.1290
epoch   5 | train loss 1.0918 | val IoU 0.0690 | val Dice 0.1290
epoch   6 | train loss 1.0344 | val IoU 0.0703 | val Dice 0.1314
epoch   7 | train loss 1.0134 | val IoU 0.0785 | val Dice 0.1456
epoch   8 | train loss 0.9397 | val IoU 0.0560 | val Dice 0.1061
epoch   9 | train loss 0.8684 | val IoU 0.0731 | val Dice 0.1363
epoch  10 | train loss 0.8083 | val IoU 0.0737 | val Dice 0.1373
epoch  11 | train loss 0.7327 | val IoU 0.0790 | val Dice 0.1465
epoch  12 | train loss 0.7054 | val IoU 0.1125 | val Dice 0.2023
epoch  13 | train loss 0.6810 | val IoU 0.0583 | val Dice 0.1103
epoch  14 | train loss 0.6242 | val IoU 0.1205 | val Dice 0.2151
epoch  15 | train loss 0.6058 | val IoU 0.1149 | val Dice 0.2060
epoch  16 | train loss 0.